In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from sqlalchemy import create_engine
import warnings
from datetime import datetime, timedelta 

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries imported Completedfully")

✅ Libraries imported Completedfully


In [3]:
# Database connection parameters
# Note: Run the connection in the terminal first:
# psql -h 127.0.0.1 -p 5432 -U dfstechbi -d db_fraud
from urllib.parse import quote_plus
from sqlalchemy import text

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': '5432',
    'database': 'db_fraud',
    'user': 'dfstechbi',
    'password': 'DfsTeChB1@923'  # You'll enter this when prompted
}

# Prompt for password
from getpass import getpass
DB_CONFIG['password'] = quote_plus(getpass('Enter database password: '))

# Create SQLAlchemy engine
connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_string)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print("✅ Database connection successful!")
        print(f"PostgreSQL version: {result.fetchone()[0].split(',')[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Database connection successful!
PostgreSQL version: PostgreSQL 17.6 on x86_64-pc-linux-gnu


In [5]:
query = f"SELECT * FROM public.stixor_iar_mbar_20250701_sample where customer_msisdn='DS2P8l39xTK/X/l2d+zSpQ==' "
df = pd.read_sql(query, engine)
df.head()

,data_date,trans_id,trans_initiate_time,customer_msisdn,trx_channel,trx_type,trx_status,ac_from,ac_to,start_balance,trx_amt,end_balance,utility_company,bill_ref_number,fee,fed,reason_type,pur_of_remit,ec,merchant_id,to_a_c_reference,to_region,to_city,to_registered_channel,to_registered_date_time,to_a_c_status,to_a_c_level,to_agent_group,to_limit_group,to_charge_profile,to_credit_dl_ml_yl,to_debit_dl_ml_yl,to_year_of_birth,to_last_modified_date_time,to_dormant_date,to_re_active_date,to_place_of_birth,to_account_type_name,to_mpin_status,to_filer,to_prov,to_year_mdob,to_gmsisdn,to_trust_level,from_a_c_reference,from_region,from_city,from_registered_channel,from_registered_date_time,from_a_c_status,from_a_c_level,from_agent_group,from_limit_group,from_charge_profile,from_credit_dl_ml_yl,from_debit_dl_ml_yl,from_year_of_birth,from_last_modified_date_time,from_dormant_date,from_re_active_date,from_place_of_birth,from_account_type_name,from_mpin_status,from_filer,from_prov,from_year_mdob,from_gmsisdn,from_trust_level
0,2025-07-01,83864992377,2025-07-01 14:14:05,DS2P8l39xTK/X/l2d+zSpQ==,NEW_JC_APP,Transfer(C2C),Completed,8MxBsdkqQXU54cIbuiIYLQ==,DS2P8l39xTK/X/l2d+zSpQ==,5230.69,1990.00,7220.69,None,None,0.00,0.00,Customer Transfer to OMNO Customer via New JC APP,None,None,None,DS2P8l39xTK/X/l2d+zSpQ==,None,None,API,2025-06-11 16:25:32,Active,Level 1,Product for Minor Accounts,Rule Profile for L1 Standard OMNO Customer,OMNO L1 for App and USSD,100000.00/300000.00/3600000.00,100000.00/300000.00/3600000.00,None,2025-06-11 16:28:27,None,None,LAHORE,Customer Account,Normal,f,Unknown,2011,miSI9l+9r/JvEVgnejjJ3w==,12,8MxBsdkqQXU54cIbuiIYLQ==,None,None,API,2025-06-21 17:21:35,Active,Level 1,Product for Minor Accounts,Rule Profile for L1 Register Customer,Charge Profile for L1 Register Customer,100000.00/300000.00/3600000.00,100000.00/300000.00/3600000.00,NaN,2025-06-21 20:07:33,None,None,LAWA CHAKWAL,Customer Account,Normal,f,Unknown,2009,JKnvGXCSO0l3OsKprFAWfg==,11
1,2025-07-01,83868076538,2025-07-01 15:03:36,DS2P8l39xTK/X/l2d+zSpQ==,NEW_JC_APP,Transfer(C2C),Completed,jaMEzVnilThCaRrfivwXyw==,DS2P8l39xTK/X/l2d+zSpQ==,7320.69,160.00,7480.69,None,None,0.00,0.00,Customer Transfer to OMNO Customer via New JC APP,None,None,None,DS2P8l39xTK/X/l2d+zSpQ==,None,None,API,2025-06-11 16:25:32,Active,Level 1,Product for Minor Accounts,Rule Profile for L1 Standard OMNO Customer,OMNO L1 for App and USSD,100000.00/300000.00/3600000.00,100000.00/300000.00/3600000.00,None,2025-06-11 16:28:27,None,None,LAHORE,Customer Account,Normal,f,Unknown,2011,miSI9l+9r/JvEVgnejjJ3w==,12,jaMEzVnilThCaRrfivwXyw==,None,None,BIO,2022-11-21 17:25:38,Active,Level 1,Product for L1 Registered Customer,Rule Profile for L1 Register Customer,Charge Profile for L1 Register Customer,100000.00/300000.00/3600000.00,100000.00/300000.00/3600000.00,2003.00,2022-11-21 17:29:43,None,None,BUREWALA VEHARI,Customer Account,Normal,f,Unknown,None,None,11


In [5]:
EVENT_DATE='2025-07-29'
query = f"""
WITH base AS (
    SELECT
        customer_msisdn,
        CAST(data_date AS DATE) AS data_date,
        trx_amt,
        trx_status,
        trx_channel,
        trx_type,
        merchant_id,
        reason_type,
        pur_of_remit,
        utility_company,
        start_balance,
        end_balance,
        trans_initiate_time,
        ac_from,
        ac_to,
        bill_ref_number,
        fee,
        fed,
        trans_id,
        ec
    FROM public.stixor_iar_jul
    -- WHERE data_date <= '{EVENT_DATE}'
    WHERE customer_msisdn = 'DS2P8l39xTK/X/l2d+zSpQ=='
),
windowed AS (
    SELECT
        *,
        data_date >= CAST('{EVENT_DATE}' AS DATE) - INTERVAL '1 day' AS win_1d,
        data_date >= CAST('{EVENT_DATE}' AS DATE) - INTERVAL '3 day' AS win_3d,
        data_date >= CAST('{EVENT_DATE}' AS DATE) - INTERVAL '7 day' AS win_7d,
        data_date >= CAST('{EVENT_DATE}' AS DATE) - INTERVAL '15 day' AS win_15d,
        data_date >= CAST('{EVENT_DATE}' AS DATE) - INTERVAL '30 day' AS win_30d,
        date_part('hour', trans_initiate_time) AS trx_hour,
        CASE
            WHEN date_part('hour', trans_initiate_time) >= 0 AND date_part('hour', trans_initiate_time) < 6 THEN 'midnight'
            WHEN date_part('hour', trans_initiate_time) >= 6 AND date_part('hour', trans_initiate_time) < 12 THEN 'morning'
            WHEN date_part('hour', trans_initiate_time) >= 12 AND date_part('hour', trans_initiate_time) < 18 THEN 'afternoon'
            ELSE 'evening'
        END AS trx_time_bucket
    FROM base
),
agg AS (
    SELECT
        ac_from,
        -- 1d window
        COUNT(*) FILTER (WHERE win_1d) AS tx_count_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_status = 'Completed') AS tx_success_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_status != 'Completed') AS tx_failed_1d,
        COUNT(DISTINCT data_date) FILTER (WHERE win_1d) AS active_days_1d,
        (MAX(trans_initiate_time) FILTER (WHERE win_1d) - MIN(trans_initiate_time) FILTER (WHERE win_1d)) AS tx_span_1d,
        SUM(trx_amt) FILTER (WHERE win_1d) AS sum_trx_amt_1d,
        AVG(trx_amt) FILTER (WHERE win_1d) AS avg_trx_amt_1d,
        MAX(trx_amt) FILTER (WHERE win_1d) AS max_trx_amt_1d,
        MIN(trx_amt) FILTER (WHERE win_1d) AS min_trx_amt_1d,
        STDDEV(trx_amt) FILTER (WHERE win_1d) AS stddev_trx_amt_1d,
        AVG(start_balance) FILTER (WHERE win_1d) AS avg_start_balance_1d,
        AVG(end_balance) FILTER (WHERE win_1d) AS avg_end_balance_1d,
        AVG(end_balance - start_balance) FILTER (WHERE win_1d) AS avg_balance_change_1d,
        SUM(end_balance - start_balance) FILTER (WHERE win_1d) AS total_balance_change_1d,
        COUNT(DISTINCT trx_channel) FILTER (WHERE win_1d) AS unique_channels_1d,
        COUNT(DISTINCT trx_type) FILTER (WHERE win_1d) AS unique_types_1d,
        COUNT(DISTINCT merchant_id) FILTER (WHERE win_1d) AS unique_merchants_1d,
        COUNT(DISTINCT reason_type) FILTER (WHERE win_1d) AS unique_reason_types_1d,
        COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_1d) AS unique_purposes_1d,
        COUNT(DISTINCT utility_company) FILTER (WHERE win_1d) AS unique_utilities_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_status != 'Completed')::float / NULLIF(COUNT(*) FILTER (WHERE win_1d), 0) AS failure_ratio_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_amt > 100000) AS high_value_count_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_1d), 0) AS high_value_ratio_1d,
        -- Time buckets (1d)
        COUNT(*) FILTER (WHERE win_1d AND trx_time_bucket = 'midnight') AS tx_count_midnight_1d,
        SUM(trx_amt) FILTER (WHERE win_1d AND trx_time_bucket = 'midnight') AS sum_trx_amt_midnight_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_time_bucket = 'morning') AS tx_count_morning_1d,
        SUM(trx_amt) FILTER (WHERE win_1d AND trx_time_bucket = 'morning') AS sum_trx_amt_morning_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_time_bucket = 'afternoon') AS tx_count_afternoon_1d,
        SUM(trx_amt) FILTER (WHERE win_1d AND trx_time_bucket = 'afternoon') AS sum_trx_amt_afternoon_1d,
        COUNT(*) FILTER (WHERE win_1d AND trx_time_bucket = 'evening') AS tx_count_evening_1d,
        SUM(trx_amt) FILTER (WHERE win_1d AND trx_time_bucket = 'evening') AS sum_trx_amt_evening_1d,
        -- Average hourly transaction count and amount (1d)
        COUNT(*) FILTER (WHERE win_1d)::float / NULLIF(24, 0) AS avg_hourly_tx_count_1d,
        SUM(trx_amt) FILTER (WHERE win_1d)::float / NULLIF(24, 0) AS avg_hourly_tx_amt_1d,
        -- Recency features (1d)
        -- Repeat for 3d, 7d, 15d, 30d windows
        COUNT(*) FILTER (WHERE win_3d) AS tx_count_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_status = 'Completed') AS tx_success_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_status != 'Completed') AS tx_failed_3d,
        COUNT(DISTINCT data_date) FILTER (WHERE win_3d) AS active_days_3d,
        (MAX(trans_initiate_time) FILTER (WHERE win_3d) - MIN(trans_initiate_time) FILTER (WHERE win_3d)) AS tx_span_3d,
        SUM(trx_amt) FILTER (WHERE win_3d) AS sum_trx_amt_3d,
        AVG(trx_amt) FILTER (WHERE win_3d) AS avg_trx_amt_3d,
        MAX(trx_amt) FILTER (WHERE win_3d) AS max_trx_amt_3d,
        MIN(trx_amt) FILTER (WHERE win_3d) AS min_trx_amt_3d,
        STDDEV(trx_amt) FILTER (WHERE win_3d) AS stddev_trx_amt_3d,
        AVG(start_balance) FILTER (WHERE win_3d) AS avg_start_balance_3d,
        AVG(end_balance) FILTER (WHERE win_3d) AS avg_end_balance_3d,
        AVG(end_balance - start_balance) FILTER (WHERE win_3d) AS avg_balance_change_3d,
        SUM(end_balance - start_balance) FILTER (WHERE win_3d) AS total_balance_change_3d,
        COUNT(DISTINCT trx_channel) FILTER (WHERE win_3d) AS unique_channels_3d,
        COUNT(DISTINCT trx_type) FILTER (WHERE win_3d) AS unique_types_3d,
        COUNT(DISTINCT merchant_id) FILTER (WHERE win_3d) AS unique_merchants_3d,
        COUNT(DISTINCT reason_type) FILTER (WHERE win_3d) AS unique_reason_types_3d,
        COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_3d) AS unique_purposes_3d,
        COUNT(DISTINCT utility_company) FILTER (WHERE win_3d) AS unique_utilities_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_status != 'Completed')::float / NULLIF(COUNT(*) FILTER (WHERE win_3d), 0) AS failure_ratio_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_amt > 100000) AS high_value_count_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_3d), 0) AS high_value_ratio_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_time_bucket = 'midnight') AS tx_count_midnight_3d,
        SUM(trx_amt) FILTER (WHERE win_3d AND trx_time_bucket = 'midnight') AS sum_trx_amt_midnight_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_time_bucket = 'morning') AS tx_count_morning_3d,
        SUM(trx_amt) FILTER (WHERE win_3d AND trx_time_bucket = 'morning') AS sum_trx_amt_morning_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_time_bucket = 'afternoon') AS tx_count_afternoon_3d,
        SUM(trx_amt) FILTER (WHERE win_3d AND trx_time_bucket = 'afternoon') AS sum_trx_amt_afternoon_3d,
        COUNT(*) FILTER (WHERE win_3d AND trx_time_bucket = 'evening') AS tx_count_evening_3d,
        SUM(trx_amt) FILTER (WHERE win_3d AND trx_time_bucket = 'evening') AS sum_trx_amt_evening_3d,
        COUNT(*) FILTER (WHERE win_3d)::float / NULLIF(72, 0) AS avg_hourly_tx_count_3d,
        SUM(trx_amt) FILTER (WHERE win_3d)::float / NULLIF(72, 0) AS avg_hourly_tx_amt_3d,
        -- 7d window
        COUNT(*) FILTER (WHERE win_7d) AS tx_count_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_status = 'Completed') AS tx_success_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_status != 'Completed') AS tx_failed_7d,
        COUNT(DISTINCT data_date) FILTER (WHERE win_7d) AS active_days_7d,
        (MAX(trans_initiate_time) FILTER (WHERE win_7d) - MIN(trans_initiate_time) FILTER (WHERE win_7d)) AS tx_span_7d,
        SUM(trx_amt) FILTER (WHERE win_7d) AS sum_trx_amt_7d,
        AVG(trx_amt) FILTER (WHERE win_7d) AS avg_trx_amt_7d,
        MAX(trx_amt) FILTER (WHERE win_7d) AS max_trx_amt_7d,
        MIN(trx_amt) FILTER (WHERE win_7d) AS min_trx_amt_7d,
        STDDEV(trx_amt) FILTER (WHERE win_7d) AS stddev_trx_amt_7d,
        AVG(start_balance) FILTER (WHERE win_7d) AS avg_start_balance_7d,
        AVG(end_balance) FILTER (WHERE win_7d) AS avg_end_balance_7d,
        AVG(end_balance - start_balance) FILTER (WHERE win_7d) AS avg_balance_change_7d,
        SUM(end_balance - start_balance) FILTER (WHERE win_7d) AS total_balance_change_7d,
        COUNT(DISTINCT trx_channel) FILTER (WHERE win_7d) AS unique_channels_7d,
        COUNT(DISTINCT trx_type) FILTER (WHERE win_7d) AS unique_types_7d,
        COUNT(DISTINCT merchant_id) FILTER (WHERE win_7d) AS unique_merchants_7d,
        COUNT(DISTINCT reason_type) FILTER (WHERE win_7d) AS unique_reason_types_7d,
        COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_7d) AS unique_purposes_7d,
        COUNT(DISTINCT utility_company) FILTER (WHERE win_7d) AS unique_utilities_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_status != 'Completed')::float / NULLIF(COUNT(*) FILTER (WHERE win_7d), 0) AS failure_ratio_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_amt > 100000) AS high_value_count_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_7d), 0) AS high_value_ratio_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_time_bucket = 'midnight') AS tx_count_midnight_7d,
        SUM(trx_amt) FILTER (WHERE win_7d AND trx_time_bucket = 'midnight') AS sum_trx_amt_midnight_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_time_bucket = 'morning') AS tx_count_morning_7d,
        SUM(trx_amt) FILTER (WHERE win_7d AND trx_time_bucket = 'morning') AS sum_trx_amt_morning_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_time_bucket = 'afternoon') AS tx_count_afternoon_7d,
        SUM(trx_amt) FILTER (WHERE win_7d AND trx_time_bucket = 'afternoon') AS sum_trx_amt_afternoon_7d,
        COUNT(*) FILTER (WHERE win_7d AND trx_time_bucket = 'evening') AS tx_count_evening_7d,
        SUM(trx_amt) FILTER (WHERE win_7d AND trx_time_bucket = 'evening') AS sum_trx_amt_evening_7d,
        COUNT(*) FILTER (WHERE win_7d)::float / NULLIF(168, 0) AS avg_hourly_tx_count_7d,
        SUM(trx_amt) FILTER (WHERE win_7d)::float / NULLIF(168, 0) AS avg_hourly_tx_amt_7d,
        -- 15d window
        COUNT(*) FILTER (WHERE win_15d) AS tx_count_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_status = 'Completed') AS tx_success_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_status != 'Completed') AS tx_failed_15d,
        COUNT(DISTINCT data_date) FILTER (WHERE win_15d) AS active_days_15d,
        (MAX(trans_initiate_time) FILTER (WHERE win_15d) - MIN(trans_initiate_time) FILTER (WHERE win_15d)) AS tx_span_15d,
        SUM(trx_amt) FILTER (WHERE win_15d) AS sum_trx_amt_15d,
        AVG(trx_amt) FILTER (WHERE win_15d) AS avg_trx_amt_15d,
        MAX(trx_amt) FILTER (WHERE win_15d) AS max_trx_amt_15d,
        MIN(trx_amt) FILTER (WHERE win_15d) AS min_trx_amt_15d,
        STDDEV(trx_amt) FILTER (WHERE win_15d) AS stddev_trx_amt_15d,
        AVG(start_balance) FILTER (WHERE win_15d) AS avg_start_balance_15d,
        AVG(end_balance) FILTER (WHERE win_15d) AS avg_end_balance_15d,
        AVG(end_balance - start_balance) FILTER (WHERE win_15d) AS avg_balance_change_15d,
        SUM(end_balance - start_balance) FILTER (WHERE win_15d) AS total_balance_change_15d,
        COUNT(DISTINCT trx_channel) FILTER (WHERE win_15d) AS unique_channels_15d,
        COUNT(DISTINCT trx_type) FILTER (WHERE win_15d) AS unique_types_15d,
        COUNT(DISTINCT merchant_id) FILTER (WHERE win_15d) AS unique_merchants_15d,
        COUNT(DISTINCT reason_type) FILTER (WHERE win_15d) AS unique_reason_types_15d,
        COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_15d) AS unique_purposes_15d,
        COUNT(DISTINCT utility_company) FILTER (WHERE win_15d) AS unique_utilities_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_status != 'Completed')::float / NULLIF(COUNT(*) FILTER (WHERE win_15d), 0) AS failure_ratio_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_amt > 100000) AS high_value_count_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_15d), 0) AS high_value_ratio_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_time_bucket = 'midnight') AS tx_count_midnight_15d,
        SUM(trx_amt) FILTER (WHERE win_15d AND trx_time_bucket = 'midnight') AS sum_trx_amt_midnight_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_time_bucket = 'morning') AS tx_count_morning_15d,
        SUM(trx_amt) FILTER (WHERE win_15d AND trx_time_bucket = 'morning') AS sum_trx_amt_morning_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_time_bucket = 'afternoon') AS tx_count_afternoon_15d,
        SUM(trx_amt) FILTER (WHERE win_15d AND trx_time_bucket = 'afternoon') AS sum_trx_amt_afternoon_15d,
        COUNT(*) FILTER (WHERE win_15d AND trx_time_bucket = 'evening') AS tx_count_evening_15d,
        SUM(trx_amt) FILTER (WHERE win_15d AND trx_time_bucket = 'evening') AS sum_trx_amt_evening_15d,
        COUNT(*) FILTER (WHERE win_15d)::float / NULLIF(360, 0) AS avg_hourly_tx_count_15d,
        SUM(trx_amt) FILTER (WHERE win_15d)::float / NULLIF(360, 0) AS avg_hourly_tx_amt_15d,
        -- 30d window
        COUNT(*) FILTER (WHERE win_30d) AS tx_count_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_status = 'Completed') AS tx_success_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_status != 'Completed') AS tx_failed_30d,
        COUNT(DISTINCT data_date) FILTER (WHERE win_30d) AS active_days_30d,
        (MAX(trans_initiate_time) FILTER (WHERE win_30d) - MIN(trans_initiate_time) FILTER (WHERE win_30d)) AS tx_span_30d,
        SUM(trx_amt) FILTER (WHERE win_30d) AS sum_trx_amt_30d,
        AVG(trx_amt) FILTER (WHERE win_30d) AS avg_trx_amt_30d,
        MAX(trx_amt) FILTER (WHERE win_30d) AS max_trx_amt_30d,
        MIN(trx_amt) FILTER (WHERE win_30d) AS min_trx_amt_30d,
        STDDEV(trx_amt) FILTER (WHERE win_30d) AS stddev_trx_amt_30d,
        AVG(start_balance) FILTER (WHERE win_30d) AS avg_start_balance_30d,
        AVG(end_balance) FILTER (WHERE win_30d) AS avg_end_balance_30d,
        AVG(end_balance - start_balance) FILTER (WHERE win_30d) AS avg_balance_change_30d,
        SUM(end_balance - start_balance) FILTER (WHERE win_30d) AS total_balance_change_30d,
        COUNT(DISTINCT trx_channel) FILTER (WHERE win_30d) AS unique_channels_30d,
        COUNT(DISTINCT trx_type) FILTER (WHERE win_30d) AS unique_types_30d,
        COUNT(DISTINCT merchant_id) FILTER (WHERE win_30d) AS unique_merchants_30d,
        COUNT(DISTINCT reason_type) FILTER (WHERE win_30d) AS unique_reason_types_30d,
        COUNT(DISTINCT pur_of_remit) FILTER (WHERE win_30d) AS unique_purposes_30d,
        COUNT(DISTINCT utility_company) FILTER (WHERE win_30d) AS unique_utilities_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_status != 'Completed')::float / NULLIF(COUNT(*) FILTER (WHERE win_30d), 0) AS failure_ratio_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_amt > 100000) AS high_value_count_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_amt > 100000)::float / NULLIF(COUNT(*) FILTER (WHERE win_30d), 0) AS high_value_ratio_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_time_bucket = 'midnight') AS tx_count_midnight_30d,
        SUM(trx_amt) FILTER (WHERE win_30d AND trx_time_bucket = 'midnight') AS sum_trx_amt_midnight_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_time_bucket = 'morning') AS tx_count_morning_30d,
        SUM(trx_amt) FILTER (WHERE win_30d AND trx_time_bucket = 'morning') AS sum_trx_amt_morning_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_time_bucket = 'afternoon') AS tx_count_afternoon_30d,
        SUM(trx_amt) FILTER (WHERE win_30d AND trx_time_bucket = 'afternoon') AS sum_trx_amt_afternoon_30d,
        COUNT(*) FILTER (WHERE win_30d AND trx_time_bucket = 'evening') AS tx_count_evening_30d,
        SUM(trx_amt) FILTER (WHERE win_30d AND trx_time_bucket = 'evening') AS sum_trx_amt_evening_30d,
        COUNT(*) FILTER (WHERE win_30d)::float / NULLIF(720, 0) AS avg_hourly_tx_count_30d,
        SUM(trx_amt) FILTER (WHERE win_30d)::float / NULLIF(720, 0) AS avg_hourly_tx_amt_30d
    FROM windowed
    GROUP BY ac_from
)
SELECT *
FROM agg
"""
df = pd.read_sql(query, engine)
df.head()

,ac_from,tx_count_1d,tx_success_1d,tx_failed_1d,active_days_1d,tx_span_1d,sum_trx_amt_1d,avg_trx_amt_1d,max_trx_amt_1d,min_trx_amt_1d,stddev_trx_amt_1d,avg_start_balance_1d,avg_end_balance_1d,avg_balance_change_1d,total_balance_change_1d,unique_channels_1d,unique_types_1d,unique_merchants_1d,unique_reason_types_1d,unique_purposes_1d,unique_utilities_1d,failure_ratio_1d,high_value_count_1d,high_value_ratio_1d,tx_count_midnight_1d,sum_trx_amt_midnight_1d,tx_count_morning_1d,sum_trx_amt_morning_1d,tx_count_afternoon_1d,sum_trx_amt_afternoon_1d,tx_count_evening_1d,sum_trx_amt_evening_1d,avg_hourly_tx_count_1d,avg_hourly_tx_amt_1d,tx_count_3d,tx_success_3d,tx_failed_3d,active_days_3d,tx_span_3d,sum_trx_amt_3d,avg_trx_amt_3d,max_trx_amt_3d,min_trx_amt_3d,stddev_trx_amt_3d,avg_start_balance_3d,avg_end_balance_3d,avg_balance_change_3d,total_balance_change_3d,unique_channels_3d,unique_types_3d,unique_merchants_3d,unique_reason_types_3d,unique_purposes_3d,unique_utilities_3d,failure_ratio_3d,high_value_count_3d,high_value_ratio_3d,tx_count_midnight_3d,sum_trx_amt_midnight_3d,tx_count_morning_3d,sum_trx_amt_morning_3d,tx_count_afternoon_3d,sum_trx_amt_afternoon_3d,tx_count_evening_3d,sum_trx_amt_evening_3d,avg_hourly_tx_count_3d,avg_hourly_tx_amt_3d,tx_count_7d,tx_success_7d,tx_failed_7d,active_days_7d,tx_span_7d,sum_trx_amt_7d,avg_trx_amt_7d,max_trx_amt_7d,min_trx_amt_7d,stddev_trx_amt_7d,avg_start_balance_7d,avg_end_balance_7d,avg_balance_change_7d,total_balance_change_7d,unique_channels_7d,unique_types_7d,unique_merchants_7d,unique_reason_types_7d,unique_purposes_7d,unique_utilities_7d,failure_ratio_7d,high_value_count_7d,high_value_ratio_7d,tx_count_midnight_7d,sum_trx_amt_midnight_7d,tx_count_morning_7d,sum_trx_amt_morning_7d,tx_count_afternoon_7d,sum_trx_amt_afternoon_7d,tx_count_evening_7d,sum_trx_amt_evening_7d,avg_hourly_tx_count_7d,avg_hourly_tx_amt_7d,tx_count_15d,tx_success_15d,tx_failed_15d,active_days_15d,tx_span_15d,sum_trx_amt_15d,avg_trx_amt_15d,max_trx_amt_15d,min_trx_amt_15d,stddev_trx_amt_15d,avg_start_balance_15d,avg_end_balance_15d,avg_balance_change_15d,total_balance_change_15d,unique_channels_15d,unique_types_15d,unique_merchants_15d,unique_reason_types_15d,unique_purposes_15d,unique_utilities_15d,failure_ratio_15d,high_value_count_15d,high_value_ratio_15d,tx_count_midnight_15d,sum_trx_amt_midnight_15d,tx_count_morning_15d,sum_trx_amt_morning_15d,tx_count_afternoon_15d,sum_trx_amt_afternoon_15d,tx_count_evening_15d,sum_trx_amt_evening_15d,avg_hourly_tx_count_15d,avg_hourly_tx_amt_15d,tx_count_30d,tx_success_30d,tx_failed_30d,active_days_30d,tx_span_30d,sum_trx_amt_30d,avg_trx_amt_30d,max_trx_amt_30d,min_trx_amt_30d,stddev_trx_amt_30d,avg_start_balance_30d,avg_end_balance_30d,avg_balance_change_30d,total_balance_change_30d,unique_channels_30d,unique_types_30d,unique_merchants_30d,unique_reason_types_30d,unique_purposes_30d,unique_utilities_30d,failure_ratio_30d,high_value_count_30d,high_value_ratio_30d,tx_count_midnight_30d,sum_trx_amt_midnight_30d,tx_count_morning_30d,sum_trx_amt_morning_30d,tx_count_afternoon_30d,sum_trx_amt_afternoon_30d,tx_count_evening_30d,sum_trx_amt_evening_30d,avg_hourly_tx_count_30d,avg_hourly_tx_amt_30d
0,065sBpaWGhGBKKaz2AVyug==,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0.00,NaN,0,0,0,0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0.00,NaN,2,2,0,1,0 days 00:35:57,400.00,200.00,200.00,200.00,0.00,3054.39,3254.39,200.00,400.00,1,1,0,1,0,0,0.00,0,0.00,0,NaN,1,200.00,1,200.00,0,NaN,0.01,2.38,2,2,0,1,0 days 00:35:57,400.00,200.00,200.00,200.00,0.00,3054.39,3254.39,200.00,400.00,1,1,0,1,0,0,0.00,0,0.00,0,NaN,1,200.00,1,200.00,0,NaN,0.01,1.11,2,2,0,1,0 days 00:35:57,400.00,200.00,200.00,200.00,0.00,3054.39,3254.39,200.00,400.00,1,1,0,1,0,0,0.00,0,0.00,0,NaN,1,200.00,1,200.00,0,NaN,0.00,0.56
1,1vqi4I347Wmh+AOmnnlc4Q==,1,1,0,1,0 days,100.00,100.00,100.00,100.00,NaN,636.39,736.39,100.00,100.00,1,1,0,1,0,0,

In [4]:
query = f"""
SELECT count(distinct ac_from)
    FROM public.stixor_iar_jul
"""
df = pd.read_sql(query, engine)
df.head()

: 

: 